# **Функции, замыкания, декораторы.**

Сегодня мы будем собирать функцию снизу вверх: сначала как ею пользоваться, потом как устроен вызов, потом где живут имена и как из этого получается декоратор.

---

## **Сегодня пройдемся по этому плану:**

1. **Основы** – зачем функция, как её объявить и вызвать, как передать аргументы.
2. **Устройство вызова** – объект, кадр, стек, рекурсия.
3. **Имена** – `LEGB`, `global` / `nonlocal`, замыкание.
4. **Декоратор** – обёртка, `wraps`, хинты.

Каждый кусок нужен следующему. Строки и файлы – не сегодня.

## **Что такое функция и зачем она.**

Кусок кода можно копировать. Через неделю в пяти местах чуть-чуть разные копии, и баг чинится только в трёх.

> **Функция** – именованный кусок кода, к которому можно обратиться из другого места программы.

Зачем она, кроме «не копировать»:

- одно место правки;
- программа читается как набор шагов с говорящими именами, а не как простыня;
- имя функции и имена параметров сами документируют намерение.

Синтаксис подождёт один абзац. Сначала вопрос, который без функции неудобен: сравнить два числа относительно их масштаба, а не «на сколько единиц больше».

### **Объявление.**

`def` создаёт функцию. Имя, скобки, двоеточие, тело с отступом.

Если тела ещё нет, а синтаксис уже нужен – в блоке пишут `pass`.

> **`pass`** – пустая инструкция. «Здесь должен быть блок, но делать пока нечего». Без неё `def` с пустым телом – синтаксическая ошибка.

In [2]:
def function():
    pass  # блок есть, действий нет

### **Вызов и возвращаемое значение.**

Имя со скобками – просьба выполнить. Результат можно положить в переменную.

В Python функция **всегда** что-то возвращает. Нет `return` – возвращается `None`.

In [3]:
result = function()
print(result)  # -> None

None


### **Осмысленный пример.**

Считаем относительную разницу: насколько `x` и `y` разъехались по сравнению со своим средним. Один вопрос – одна функция.

In [ ]:
def relative_difference(x, y):
    delta = x - y
    mean = (x + y) / 2
    return abs(delta / mean)


print(relative_difference(3, 5))  # -> 0.5

### **Аргументы: по позиции и по имени.**

При вызове значения подставляются в параметры. По порядку – позиционно. По имени – тогда порядок не важен.

> **PEP 8.** Вокруг `=` при передаче по имени пробелы не ставят: `y=5`, не `y = 5`.

In [ ]:
print(relative_difference(3, 5))       # оба позиционно
print(relative_difference(x=3, y=5))   # оба по имени, порядок свободный
print(relative_difference(y=5, 3))     # сначала позиция, потом имя

Имя, а потом позиция – нельзя. Python уже не знает, какой слот ещё свободен.

In [ ]:
# relative_difference(x=3, 5)  # SyntaxError: positional argument follows keyword argument

Та же функция ломается, если среднее равно нулю: `x` и `-x`. Делить на ноль нельзя – Python бросает `ZeroDivisionError`.

In [ ]:
# relative_difference(5, -5)  # ZeroDivisionError: float division by zero

### **Несколько `return`. Вызов заканчивается сразу.**

Сначала поправим в лоб: посчитали среднее, если оно почти ноль – кладём `None` в переменную, иначе – отношение. Один `return` в конце.

In [ ]:
def relative_difference(x, y):
    delta = x - y
    mean = (x + y) / 2
    if abs(mean) < 1e-10:
        res = None
    else:
        res = abs(delta / mean)
    return res


print(relative_difference(5, -5))  # -> None
print(relative_difference(3, 5))   # -> 0.5

Тот же смысл короче: функция может содержать несколько `return`. Первый сработавший *заканчивает* вызов, дальше тело не идёт.

In [ ]:
def relative_difference(x, y):
    delta = x - y
    mean = (x + y) / 2
    if abs(mean) < 1e-10:
        return None  # вызов закончен, до нижней строки не дойдём
    return abs(delta / mean)


print(relative_difference(5, -5))  # -> None
print(relative_difference(3, 5))   # -> 0.5

### **Документация и аннотации.**

Сразу под `def` – строка в тройных кавычках. Это `__doc__`. Её печатает `help`.

После параметров и после `->` – подсказки типов. Python их **не проверяет** в рантайме. Это записка для людей, `help` и тайпчекера.

> **`help(объект)`** – печатает справку: подпись, документацию. Особенно удобно в интерактивном режиме.

In [4]:
def relative_difference(x: float, y: float) -> float | None:
    '''
    Сравнивает две величины с учётом их масштаба.
    '''
    delta = x - y
    mean = (x + y) / 2
    if abs(mean) < 1e-10:
        return None
    return abs(delta / mean)


help(relative_difference)
help(pow)  # то же самое работает для встроенных функций

Help on function relative_difference in module __main__:

relative_difference(x: float, y: float) -> float | None
    Сравнивает две величины с учётом их масштаба.

Help on built-in function pow in module builtins:

pow(base, exp, mod=None)
    Equivalent to base**exp with 2 arguments or base**exp % mod with 3 arguments

    Some types, such as ints, are able to use a more efficient algorithm when
    invoked using the three argument form.



### **Значение по умолчанию.**

Параметр можно не передавать, если у него есть запас. Запас пишут в `def` после `=`.

> **PEP 8.** Если у параметра есть и тип, и дефолт, вокруг `=` *ставят* пробелы: `verbose: bool = False`.

In [5]:
def relative_difference(
    x: float,
    y: float,
    verbose: bool = False,
) -> float | None:

    delta = x - y
    if verbose:
        print(f"delta: {delta}")
    mean = (x + y) / 2
    if verbose:
        print(f"mean: {mean}")
    if abs(mean) < 1e-10:
        return None
    return abs(delta / mean)


# print(relative_difference(-10, 3))
print(relative_difference(-10, 3, verbose=True))

delta: -13
mean: -3.5
3.7142857142857144


### **Функция – объект.**

`def` кладёт в имя не «команду», а объект. Без скобок мы говорим про объект. Со скобками – про результат вызова.

Поэтому объект можно положить в другое имя и передать в другую функцию.

In [7]:
def meaning_of_life() -> int:
    return 42


answer = meaning_of_life  # без скобок: второе имя той же функции. В ссылочной модели answer будет указывать
# тот же объект, что и meaning_of_life

print(answer())           # -> 42
print(answer is meaning_of_life)  # -> True 
# is проверяет, что идет ссылание на один объект, те у переменных один id

42
True


Локальные имена после вызова уже недоступны. Они жили только *во время* выполнения – к этому вернёмся, когда появится кадр.

Пока достаточно факта: снаружи имени из тела нет.

In [8]:
def some_function() -> int:
    inside = 15
    return inside


print(some_function())  # -> 15
# print(inside)  # NameError: имени inside снаружи нет

15


Раз функция – объект, её тип в подсказке – «вызываемое».

> **`Callable[[типы аргументов], тип возврата]`** – объект, который можно вызвать с такими аргументами и получить такой результат.

Так мы передаём *правило нормировки* в `relative_difference`, а не зашиваем среднее навсегда.

- Зачем вообще Callable? Чтобы в аннотации сказать: «сюда передаётся не число, не строка, а что-то, что можно вызвать скобочками» — то есть функция.



```text
Callable[ [float, float],  float ]
          └──────┬──────┘  └──┬──┘
                 │            │
        что принимает         что возвращает
        (2 аргумента,         (одно число
         оба float)            с плавающей точкой)
```

In [9]:
from collections.abc import Callable

def relative_difference(
    x: float,
    y: float,
    normalize_by: Callable[[float, float], float],# функция-аргумент
) -> float | None:                 # возвращает float ИЛИ None

    delta = x - y
    norm = normalize_by(x, y)
    if abs(norm) < 1e-10:
        return None
    return abs(delta / norm)


print(relative_difference(-5, 8, normalize_by=max))  # -> 1.625

1.625


Нормировку можно не передавать. Тогда параметр опционален: запас `None`, внутри проверяем.

> **PEP 8.** С `None` сравнивают через `is` / `is not`: это один-единственный объект, не «похожее значение».
> Предпочтительнее `is not None`, чем `not ... is None`.

In [ ]:
def relative_difference(
    x: float,
    y: float,
    normalize_by: Callable[[float, float], float] | None = None,
) -> float | None:
    delta = x - y
    if normalize_by is not None:
        norm = normalize_by(x, y)
    else:
        norm = 1.0
    if abs(norm) < 1e-10:
        return None
    return abs(delta / norm)


def mean(x: float, y: float) -> float:
    return (x + y) / 2


print(relative_difference(-5, 8))                    # -> 13.0, норма 1
print(relative_difference(-5, 8, normalize_by=mean))  # -> ~8.67
print(relative_difference(-5, 8, mean))               # то же позиционно

### **Только по имени: `*`.**

Позиционно `mean` легко перепутать с третьим числом. Звёздочка в списке параметров говорит: всё *правее* передаётся только по имени.

> **Именованный только (`*`)** – параметры после звёздочки нельзя передать позиционно.

In [ ]:
def relative_difference(
    x: float,
    y: float,
    *,
    normalize_by: Callable[[float, float], float] | None = None,
) -> float | None:
    delta = x - y
    norm = normalize_by(x, y) if normalize_by is not None else 1.0
    if abs(norm) < 1e-10:
        return None
    return abs(delta / norm)


# relative_difference(-5, 8, mean)  # TypeError: takes 2 positional arguments but 3 were given
print(relative_difference(-5, 8, normalize_by=mean))

### **Только позиционно: `/`.**

Обратное ограничение: параметры *левее* слэша нельзя передать по имени. Так устроены некоторые встроенные: имя параметра – деталь реализации, не часть контракта.

> **Позиционный только (`/`)** – параметры до слэша передаются только по порядку.

In [ ]:
def relative_difference(
    x: float,
    y: float,
    /,
    *,
    normalize_by: Callable[[float, float], float] | None = None,
) -> float | None:
    delta = x - y
    norm = normalize_by(x, y) if normalize_by is not None else 1.0
    if abs(norm) < 1e-10:
        return None
    return abs(delta / norm)


# print(relative_difference(-5, 8, normalize_by=mean))
# relative_difference(-5, y=8, normalize_by=mean)
# TypeError: positional-only arguments passed as keyword arguments: 'y'

### **Сколько угодно позиционных: `*args`.**

Пока числа удобно собрать в список и передать одним аргументом.

In [ ]:
def root_mean_square(args: list[float]) -> float:
    if not args:
        return 0.0
    mean = sum(x ** 2 for x in args) / len(args)
    return mean ** 0.5


print(root_mean_square([4, 8, 15, 16, 23, 42]))  # -> ~21.81

Вызов `root_mean_square(4, 8, 15)` так не сделать: функция ждёт *один* список. Звёздочка перед именем собирает «лишние» позиционные в кортеж – скобки вокруг списка в вызове больше не нужны.

> **`*args`** – кортеж позиционных аргументов, которые не попали в обычные параметры.
>
> **PEP 8.** В аннотации `*args` указывают тип *одного элемента*, не всего кортежа: `*args: float`.

In [ ]:
a, b = 1,2

a,b = b,a

In [ ]:
def func(a: int) -> int:
    return a

print(func('123'))

In [ ]:
def root_mean_square(a, *args: float, hmmm, aasdasd) -> float:
    print(a, args)
    if not args:
        return 0.0
    mean = sum(x ** 2 for x in args) / len(args)
    return mean ** 0.5


print(root_mean_square(3, 4, 8, 15, 16, 23, 42, hmmm=3, aasdasd=3))  # -> ~21.81

### **Сколько угодно имён: `**kwargs`.**

Две звёздочки собирают лишние именованные в словарь.

> **`**kwargs`** – словарь `имя → значение` для именованных аргументов, которые не попали в обычные параметры.

In [ ]:
def root_mean_square(*args: float, **kwargs: object) -> float:
    print(args, kwargs)
    
    verbose = bool(kwargs.get("verbose", False))
    if not args:
        if verbose:
            print("пустой список")
        return 0.0
    mean = sum(x ** 2 for x in args) / len(args)
    if verbose:
        print(f"mean square: {mean}")
    return mean ** 0.5


print(root_mean_square(4, 8, 15, 16, 23, 42, verbose="True"))

### **Распаковка: звёздочки в вызове.**

В объявлении `*` собирает. В вызове – наоборот, разбрасывает: список в позиционные, словарь в именованные.

In [ ]:
def show(x, y, /, *, option1=None, option2=None) -> None:
    print(x, y, option1, option2)


positional = [4, 8]
key_value = {"option1": 15, "option2": 16}
show(*positional, **key_value)  # -> 4 8 15 16
# show(4, 8, option1=3, **key_value)  # TypeError: multiple values for argument 'option1'

Смешаем всё в одной подписи и посмотрим, куда падает «лишнее». `True` без имени не станет `binary=True`: после `*files` позиционное значение уезжает в кортеж файлов.

In [ ]:
def read_and_merge_files(
    base_path: str,
    /,
    *files: str,
    binary: bool = False,
    **kwargs: object,
) -> None:
    print(f"{base_path=}, {files=}, {binary=}, {kwargs=}")


read_and_merge_files("a", "b", "c", "d")
read_and_merge_files("a", "b", "c", "d", True)          # True попал в files
read_and_merge_files("a", "b", "c", "d", binary=True)    # так задумано
read_and_merge_files("a", "b", "c", "d", base_path="x")  # лишний base_path в kwargs

### **`lambda`.**

Тот же объект функции, только без `def` и без имени. Тело – одно выражение, не блок.

> **Антипаттерн.** Сохранять лямбду в переменную: тогда нужен обычный `def` с именем, документацией и типами.

Место лямбды – короткий ключ сортировки, который больше нигде не нужен.

In [ ]:
lambda x: x ** 2


In [ ]:
print(sorted(["привет", "как", "дела"], key=lambda s: len(s)))  # -> ['как', 'дела', 'привет']


def string_length(s: str) -> int:
    return len(s)

Дальше вопрос не «как вызвать», а *что происходит в памяти*, когда мы вызываем.

---

## **Устройство вызова: объект, кадр, стек.**

Локальные имена пропадают после `return`. Куда они делись? Они жили в **кадре** – карточке одного незавершённого вызова.

> **Объект функции** – то, что создаёт `def`: тело, имена параметров, значения по умолчанию.
> **Кадр (frame)** – одно выполнение этой функции. Появляется на вызове, исчезает на `return`.
> **Стек вызовов** – стопка кадров. Верхний выполняется сейчас. Вызов кладёт кадр, `return` снимает.

Один кадр = один вызов, который ещё не закончился. Два вызова одной функции – два кадра, один объект.

<img src="../../Other/images/func_stacks.png" alt="Стек вызовов" width="320">

Низ стопки – кто всех позвал. На картинке это `main`, в Jupyter тот же слой называется `<module>`. Верх – кадр, который крутится сейчас.

In [ ]:
def add(a, b=10):
    c = a + b
    return c


print(add.__code__.co_varnames)  # -> ('a', 'b', 'c')  локальные имена
print(add.__defaults__)          # -> (10,)  запас на объекте, один на всех

`__code__` – скомпилированное тело. `co_varnames` – локальные имена (сначала аргументы). `__defaults__` лежит *на объекте*, не в кадре.

`inspect.stack()` показывает ту же стопку, что на картинке: нулевой элемент – верх, кадр, который выполняется сейчас.

In [ ]:
import inspect


def names_on_stack():
    return [info.function for info in inspect.stack()]


def bottom():
    print(names_on_stack()[:8])


def middle():
    bottom()


def top():
    middle()


top()
# -> ['names_on_stack', 'bottom', 'middle', 'top', ... и служебные кадры Jupyter]

### **Рекурсия – тот же объект, новые кадры.**

> **Рекурсия** – функция вызывает сама себя. Отдельного синтаксиса нет: на стек кладётся ещё один кадр той же функции.

У каждого кадра свои локальные имена. `n` в одном вызове не затирает `n` в другом.

> **Базовый случай** – ветка без рекурсивного вызова. Без неё стек растёт до `RecursionError`.

In [ ]:
def countdown(n):
    print("вход", n)
    if n == 0:
        print("дно")
        return
    countdown(n - 1)
    print("выход", n)


countdown(3)
# вход растёт стек, выход снимает кадры

Тот же стек, но кадр ещё и *отдаёт значение наверх*: `factorial(5)` ждёт `factorial(4)`, тот – `factorial(3)`, пока не сработает базовый случай.

In [ ]:
def factorial(n):
    if n <= 1:
        return 1
    return n * factorial(n - 1)


print(factorial(5))  # -> 120

`sys.getrecursionlimit()` – потолок числа кадров, не «высота нашей задачи». `sys.setrecursionlimit` как решение не используем: если глубина растёт линейно со входом, обычно нужен цикл.

In [ ]:
import sys

print(sys.getrecursionlimit())  # -> типично 1000


def depth(n=0):
    try:
        return depth(n + 1)
    except RecursionError:
        return n


print(depth())  # около лимита, часть съел Jupyter

### **Почему `list=[]` в дефолте – один список на всех.**

Запас создаётся на `def` и лежит на объекте. Кадр при вызове только берёт ссылку. Если запас – список, все вызовы без аргумента смотрят на один список.

Это та же пара «объект vs кадр», не отдельная магия `=`.

In [ ]:
A = [1,2,3]

B = A
B.append(4)

print(A)
print(B)


[1, 2, 3, 4]
[1, 2, 3, 4]


In [ ]:
A = []

def add_hi(items=[]):
    items.append("Hi!")
    return items


# print(add_hi())  # -> ['Hi!']
# print(add_hi.__defaults__)  # -> (['Hi!'],)  запас на объекте уже вырос

Второй вызов без аргумента. Что выведет – и почему `__defaults__` уже не пустой список?

In [64]:
print(add_hi(['Alexandr']))  # что выведет?

['Alexandr', 'Hi!']


Чинят так: запас – `None`, список создают *внутри кадра*.

In [ ]:
def add_hi(items=None):
    if items is None:
        items = []
    items.append("Hi!")
    return items


print(add_hi())
print(add_hi())

Если `None` – законное значение «ничего не делать», дефолт должен быть другим объектом.

> **Sentinel** – отдельный объект «значение не передали». Обычно `object()`. Нужен, когда `None` уже значит что-то своё.

In [ ]:
missing = object()


def add_hi(items=missing):
    if items is None:
        return None  # явно сказали «ничего»
    if items is missing:
        items = []   # вызвали без аргумента
    items.append("Hi!")
    return items


print(add_hi())      # -> ['Hi!']
print(add_hi(None))  # -> None

---

## **Где живут имена.**

Таблица имён вызова – поля кадра. Цикл и `if` кадр не создают: они крутятся в том же кадре, поэтому переменная из `for` остаётся снаружи.

Поиск имени идёт по маршруту `LEGB`.

> **`LEGB`** – Local (текущий кадр) → Enclosing (кадр внешней функции) → Global (модуль) → Built-in (`len`, `print`).

<img src="../../Other/images/python_namespace.png" alt="Пространства имён LEGB" width="420">

Ищем из центра наружу. Local и Enclosing на картинке в одном круге: оба живут в кадрах функций, не в модуле. Разница – *чей* кадр.

In [10]:
k = "снаружи"

for k in range(3):
    in_for = k

print(in_for)  # -> 2
print(k)       # -> 2, строка потерялась: for не создал кадр

2
2


Своего имени в кадре нет – идём дальше по `LEGB`. `label` находится в модуле, `len` – среди встроенных.

In [26]:
M = '234'


def show():
    global M 
    M = '2'
    return M


print(show())

print(M)
# print(len([1, 2, 3]))  # built-in

2
2


### **Вложенные функции.**

Функцию можно объявить внутри функции. Внутренняя *читает* имя внешней сама: своего нет – шаг E в `LEGB`.

In [ ]:
def outer():
    label = "enclosing"

    def inner():
        print(label)

    inner()


outer()  # -> enclosing

Чтение enclosing уже работает. Запись в то же имя без объявления делает его *локальным на весь кадр* – Python не дойдёт до внешней `counter`.

In [ ]:
counter = 0


def bump_broken():
    counter = counter + 1  # UnboundLocalError: локальный counter ещё не создан


# bump_broken()

Чтобы писать во внешнюю функцию, нужно сказать это явно: `nonlocal`. Имя остаётся в enclosing-кадре, внутренняя его меняет.

In [ ]:
def make_counter():
    count = 0

    def bump():
        nonlocal count
        count = count + 1
        return count

    return bump


clicks = make_counter()
print(clicks())  # -> 1
print(clicks())  # -> 2

`make_counter` уже вернулась, а `clicks` знает свой `count`. Это замыкание.

---

## **Замыкание.**

> **Замыкание** – функция вместе с именами, которые она взяла снаружи. Внешняя уже закончилась, внутренняя эти имена всё ещё видит.

Следует из двух вещей: функция – объект, работает `LEGB`.

In [ ]:
def make_multiplier(factor):
    def multiply(value):
        return value * factor

    return multiply


double = make_multiplier(2)
triple = make_multiplier(3)
print(double(5), triple(5))  # -> 10 15
print(double.__closure__[0].cell_contents)  # -> 2, захваченный factor

`__closure__` показывает, что именно захватили. На каждый день это не нужно – достаточно понимать, что имена живут дольше внешней функции.

---

## **Декоратор.**

Хотим добавить поведение вокруг функции, не трогая тело и все места вызова.

> **Декоратор** – функция, которая принимает функцию и возвращает новую с тем же способом вызова.

Сначала без `@`: подменили имя, как `answer = meaning_of_life`.

In [ ]:
def logger(func):
    def wrapper(*args, **kwargs):
        print("вызываем", func.__name__)
        result = func(*args, **kwargs)
        print("готово")
        return result

    return wrapper


def add(a, b):
    return a + b


add = logger(add)
print(add(2, 3))

```python
@logger
def add(a, b):
    return a + b
```

это ровно `add = logger(add)` после объявления.

In [ ]:
@logger
def mul(a, b):
    return a * b


print(mul(3, 4))

### **Имя, документация и `wraps`.**

У функции есть `__name__`, `__doc__`, аннотации типов. Их читают `help` и IDE.

> **`__annotations__`** – словарь подсказок типов. Python их не проверяет.

Обёртка – другая функция: имя `wrapper`, пустые аннотации, подпись `(*args, **kwargs)`.

In [ ]:
def add(a: int, b: int = 0) -> int:
    '''Складывает два числа.'''
    return a + b


print(add.__name__, add.__doc__)
print(add.__annotations__)
print(inspect.signature(add))

После `@logger` имя `add` указывает на `wrapper`. `__name__` и документация – уже обёртки.

In [ ]:
@logger
def add(a: int, b: int = 0) -> int:
    '''Складывает два числа.'''
    return a + b


print(add.__name__)            # -> wrapper
print(add.__annotations__)     # -> {}
print(inspect.signature(add))  # -> (*args, **kwargs)

Вручную копировать `__name__` мало: пропадут docstring и аннотации.

> **`functools.wraps(func)`** – декоратор на обёртку. Копирует имя, документацию и аннотации исходной функции, исходную кладёт в `__wrapped__`.

In [ ]:
from functools import wraps


def logger(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print("вызываем", func.__name__)
        return func(*args, **kwargs)

    return wrapper


@logger
def add(a: int, b: int = 0) -> int:
    '''Складывает два числа.'''
    return a + b


print(add.__name__, inspect.signature(add))

`wraps` чинит `help` и `__name__`. Тайпчекер по-прежнему видит `wrapper(*args, **kwargs)` – это чинят через `ParamSpec`, не сегодня.

Живое замыкание в декораторе: вызвать исходную функцию один раз, дальше отдавать сохранённое.

In [ ]:
def once(func):
    called = False
    saved = None

    @wraps(func)
    def wrapper(*args, **kwargs):
        nonlocal called, saved
        if not called:
            saved = func(*args, **kwargs)
            called = True
        return saved

    return wrapper


@once
def load_config():
    print("читаем конфиг")
    return {"debug": True}


print(load_config())
print(load_config())

### **Декоратор с параметром.**

`@repeat(3)` сначала вызывает `repeat(3)` – получается декоратор – и уже он оборачивает функцию. Три этажа: параметр → декоратор → обёртка.

In [ ]:
def repeat(times):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            result = None
            for _ in range(times):
                result = func(*args, **kwargs)
            return result

        return wrapper

    return decorator


@repeat(3)
def beep():
    print("beep")


beep()

## **Дополнительно почитать.**

Не обязательно до следующей пары.

- [Функции Python – это объекты первого класса](https://habr.com/ru/companies/otus/articles/725374/) – ту же цепочку другими словами: имя, передача, вложенность, замыкание. Кусок про `__call__` у класса – не сегодня.
- [Устройство CPython](https://habr.com/ru/companies/yandex/articles/511972/) – русским текстом про объект кода, кадр и стек. Дальше по статье (память, генераторы, `GIL`) – другие пары; доклад 2020 года, с 3.11 кадр в CPython устроен иначе, идея та же.
- [Function objects](https://docs.python.org/3/reference/datamodel.html#function-objects) в data model – официально: чем объект функции отличается от объекта кода и от кадра. Дефолты живут на функции, не в коде. Рядом на той же странице – [code objects](https://docs.python.org/3/reference/datamodel.html#code-objects) и [frame objects](https://docs.python.org/3/reference/datamodel.html#frame-objects).

На следующей паре – итераторы, генераторы, файлы.